# Dashboard KPI's — Bloc 5

Ce notebook charge les tables **Silver** produites par ton ETL (`data/base.db`)
et génère un graphique Plotly Express pour chaque KPI calculé dans la partie
*Transformation* du script.

> ⚠️ **Avant d'exécuter ce notebook**, relance ton ETL avec la petite correction
> ci-dessous (voir cellule suivante) : plusieurs `groupby(...).agg(...)` sont
> stockés **sans `reset_index()`**, et comme le chargement Silver fait
> `to_sql(..., index=False)`, l'index (souvent `month`, `factory_id` ou
> `machine_id`) est **perdu** lors de l'écriture en base. Sans cette clé, les
> graphiques ci-dessous ne pourront pas être tracés (colonne manquante).


## 🔧 Fix recommandé dans ton ETL

Remplace ta fonction `to_dataframe` par cette version, qui remet automatiquement
en colonne tout index qui n'est pas un `RangeIndex` par défaut (donc tous les
`groupby`/`resample`/`set_index` oubliés) :

```python
def to_dataframe(obj, object_name):
    if isinstance(obj, pd.DataFrame):
        obj = obj.copy()
        if not isinstance(obj.index, pd.RangeIndex):
            obj = obj.reset_index()
        return obj

    if isinstance(obj, pd.Series):
        return obj.rename(object_name).reset_index()

    if isinstance(obj, dict):
        return pd.DataFrame([obj])

    if isinstance(obj, (list, tuple)):
        return pd.DataFrame(obj)

    return pd.DataFrame({"value": [obj]})
```

Tables concernées par ce bug dans ton script actuel : `silver_global_sales`,
`silver_facto_prod_elec_cost`, `silver_high_severity_total`,
`silver_nb_pannes_global`, `silver_nb_pannes_high`, `silver_nb_pannes_medium`,
`silver_nb_pannes_low`, `silver_alerts_machines`.

Relance ensuite ton ETL, puis exécute ce notebook.


In [16]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine

engine = create_engine("sqlite:///data/base.db")

def load(table):
    return pd.read_sql(f"SELECT * FROM {table}", con=engine)

def first_existing(df, candidates, fallback=None):
    for c in candidates:
        if c in df.columns:
            return c
    return fallback

import sqlalchemy as sa
insp = sa.inspect(engine)
silver_tables = sorted(t for t in insp.get_table_names() if t.startswith("silver_"))
print(f"{len(silver_tables)} tables silver trouvées :")
for t in silver_tables:
    print(" -", t)


23 tables silver trouvées :
 - silver_alerts_machines
 - silver_all_costs_2025
 - silver_all_global_cost
 - silver_benef_produit_month
 - silver_ca_month
 - silver_conso_by_facto
 - silver_date_pannes
 - silver_df_benef
 - silver_df_ca
 - silver_facto_prod_elec_cost
 - silver_global_benef
 - silver_global_sales
 - silver_high_severity_total
 - silver_machines
 - silver_monthly_mttr
 - silver_nb_pannes_global
 - silver_nb_pannes_high
 - silver_nb_pannes_low
 - silver_nb_pannes_medium
 - silver_renta_produit
 - silver_sorted_sales
 - silver_to_resolve_count
 - silver_top_clients


## 1. Rentabilité par produit
`benef_product`, `rentability_product` — table `silver_renta_produit`

In [17]:
df = load("silver_renta_produit")
name_col = first_existing(df, ["product_name", "name", "product_id"])

agg = df.groupby(name_col, as_index=False).agg(
    benef_product=("benef_product", "sum"),
    rentability_product=("rentability_product", "mean"),
    quantity=("quantity", "sum"),
)

fig = px.bar(
    agg.sort_values("benef_product", ascending=False),
    x=name_col, y="benef_product",
    title="Bénéfice total par produit",
    labels={name_col: "Produit", "benef_product": "Bénéfice (€)"},
    text_auto=".2s",
)
fig.show()

fig2 = px.bar(
    agg.sort_values("rentability_product", ascending=False),
    x=name_col, y="rentability_product",
    title="Rentabilité moyenne par unité vendue",
    labels={name_col: "Produit", "rentability_product": "Rentabilité (€/unité)"},
    text_auto=".2f",
    color="rentability_product",
    color_continuous_scale="RdYlGn",
)
fig2.show()


## 2. Top clients
Table `silver_top_clients`

In [18]:
df = load("silver_top_clients")
name_col = first_existing(df, ["client_name", "name", "company_name", "client_id"])

top = df.sort_values("revenue_eur", ascending=False).head(15)

fig = px.bar(
    top, x="revenue_eur", y=name_col, orientation="h",
    title="Top 15 clients par chiffre d'affaires",
    labels={"revenue_eur": "CA (€)", name_col: "Client"},
    text_auto=".2s",
)
fig.update_layout(yaxis=dict(categoryorder="total ascending"))
fig.show()


## 3. CA cumulé par client dans le temps
`ca_cumule_client` — table `silver_sorted_sales`

In [19]:
df = load("silver_sorted_sales")
df2 = load("bronze_client")
df = df.merge(df2[["client_id", "client_name"]])
df["month"] = pd.to_datetime(df["month"])

# on limite aux 8 plus gros clients pour la lisibilité
top_ids = (
    df.groupby("client_id")["ca_cumule_client"].max()
    .sort_values(ascending=False).head(8).index
)

plot_df = df[df["client_id"].isin(top_ids)]

fig = px.line(
    plot_df, x="month", y="ca_cumule_client", color="client_name",
    title="Chiffre d'affaires cumulé par client (top 8)",
    labels={"month": "Mois", "ca_cumule_client": "CA cumulé (€)", "client_id": "Client"},
    markers=True,
)
fig.show()


## 4. Saisonnalité des ventes globales
Table `silver_global_sales` (nécessite le fix `reset_index`)

In [20]:
df = load("silver_global_sales")


df["month"] = pd.to_datetime(df["month"])



fig = px.line(
    df, x="month", y="revenue_eur",
    title="CA global par mois — saisonnalité",
    labels={"month": "Mois", "revenue_eur": "CA (€)"},
    markers=True,
)
fig.show()

fig2 = px.bar(
    df, x="month", y="quantity",
    title="Quantités vendues par mois",
    labels={"month": "Mois", "quantity": "Quantité"},
)
fig2.show()


## 5. Consommation et coûts énergétiques par usine
Tables `silver_facto_prod_elec_cost` (fix requis) et `silver_conso_by_facto`

In [ ]:
df_elec = load("silver_facto_prod_elec_cost")
df_usine = load("bronze_usine")

df_elec = df_elec.merge(df_usine[["factory_id", "factory_name"]], how="left", on="factory_id")

fig = px.bar(
    df_elec.sort_values("energy_consumption_kwh", ascending=False),
    x="factory_name", y="energy_consumption_kwh",
    title="Consommation d'énergie totale par usine",
    labels={"factory_name": "Usine", "energy_consumption_kwh": "Conso (kWh)"},
    text_auto=".2s",
)
fig.show()

df_conso = load("silver_conso_by_facto")
df_conso["month"] = pd.to_datetime(df_conso["month"])
df_conso = df_conso.merge(df_usine[["factory_id", "factory_name"]], how="left", on="factory_id")


fig2 = px.line(
    df_conso, x="month", y="conso_cost", color="factory_name",
    title="Coût énergétique mensuel par usine",
    labels={"month": "Mois", "conso_cost": "Coût énergie (€)", "factory_name": "Usine"},
    markers=True,
)
fig2.show()


## 6. Coûts totaux par usine (maintenance + énergie + pièces + qualité)
Table `silver_all_costs_2025`

In [34]:
df

,factory_id,total_cost_facto,factory_name,country,city,machine_count,_silver_load_timestamp_utc
0,1,2.015255e+06,Usine Lyon,France,Lyon,20,2026-08-11T07:10:40.378147+00:00
1,2,1.877002e+06,Usine Lille,France,Lille,18,2026-08-11T07:10:40.378147+00:00
2,3,2.045594e+06,Usine Toulouse,France,Toulouse,22,2026-08-11T07:10:40.378147+00:00
3,4,2.430820e+06,Usine Hamburg,Germany,Hamburg,25,2026-08-11T07:10:40.378147+00:00
4,5,3.746966e+06,Usine Turin,Italy,Turin,19,2026-08-11T07:10:40.378147+00:00


,factory_id,total_cost_facto,factory_name,country,city,machine_count,_silver_load_timestamp_utc
0,1,2.015255e+06,Usine Lyon,France,Lyon,20,2026-08-11T07:25:33.942838+00:00
1,2,1.877002e+06,Usine Lille,France,Lille,18,2026-08-11T07:25:33.942838+00:00
2,3,2.045594e+06,Usine Toulouse,France,Toulouse,22,2026-08-11T07:25:33.942838+00:00
3,4,2.430820e+06,Usine Hamburg,Germany,Hamburg,25,2026-08-11T07:25:33.942838+00:00
4,5,3.746966e+06,Usine Turin,Italy,Turin,19,2026-08-11T07:25:33.942838+00:00


In [41]:
df = load("silver_all_costs_2025")
df_all_costs = load("silver_all_costs_facto")
name_col = first_existing(df, ["factory_name", "name", "factory_id"])

fig = px.bar(
    df.sort_values("total_cost_facto", ascending=False),
    x=name_col, y="total_cost_facto",
    title="Coût total annuel par usine",
    labels={name_col: "Usine", "total_cost_facto": "Coût total (€)"},
    text_auto=".2s",
    color="country" if "country" in df.columns else None,
)
fig.show()

df_all_costs = df_all_costs.merge(df[["factory_id", "factory_name"]], how="left", on="factory_id")

# Décomposition des coûts par usine (barres empilées)
cost_cols = [c for c in ["maintenance_cost_eur", "conso_cost", "yearly_avg_cost", "rework_cost_eur"] if c in df_all_costs.columns]
melted = df_all_costs.melt(id_vars=[name_col], value_vars=cost_cols, var_name="type_cout", value_name="montant")
melted = melted.rename(columns={"maintenance_cost_eur" : "coût de maintenance", "conso_cost" : "coûts de consommation","yearly_avg_cost" : "côut moyen annuel des pièces", "rework_cost_eur" : "coûts liés à la qualité"  })

fig2 = px.bar(
    melted, x=name_col, y="montant", color="type_cout",
    title="Décomposition des coûts par usine",
    labels={name_col: "Usine", "montant": "Montant (€)", "type_cout": "Type de coût"},
)
fig2.show()


## 7. Coûts globaux mensuels (énergie, maintenance, pièces, qualité)
Table `silver_all_global_cost`

In [42]:
df = load("silver_all_global_cost")
df["month"] = pd.to_datetime(df["month"])

cost_cols = [c for c in ["conso_cost", "maintenance_cost_eur", "avg_global_piece_cost", "rework_cost_eur"] if c in df.columns]
melted = df.melt(id_vars=["month"], value_vars=cost_cols, var_name="type_cout", value_name="montant")

fig = px.area(
    melted, x="month", y="montant", color="type_cout",
    title="Répartition mensuelle des coûts globaux",
    labels={"month": "Mois", "montant": "Montant (€)", "type_cout": "Type de coût"},
)
fig.show()

fig2 = px.line(
    df, x="month", y="global_cost",
    title="Coût global mensuel (total)",
    labels={"month": "Mois", "global_cost": "Coût total (€)"},
    markers=True,
)
fig2.show()


## 8. Bénéfice mensuel (CA - coûts)
Table `silver_df_benef` et KPI annuel `silver_global_benef`

In [46]:
melted

,month,indicateur,montant
0,2025-01-01,revenue_eur,7.377582e+05
1,2025-02-01,revenue_eur,1.166386e+06
2,2025-03-01,revenue_eur,1.271305e+06
3,2025-04-01,revenue_eur,1.363165e+06
4,2025-05-01,revenue_eur,9.626214e+05
5,2025-06-01,revenue_eur,9.774393e+05
6,2025-07-01,revenue_eur,1.141239e+06
7,2025-08-01,revenue_eur,6.384785e+05
8,2025-09-01,revenue_eur,1.454136e+06
9,2025-10-01,revenue_eur,1.062819e+06


In [50]:
df = load("silver_df_benef")
df["month"] = pd.to_datetime(df["month"])

melted1 = df.melt(id_vars=["month"], value_vars=["revenue_eur", "global_cost"],
                  var_name="indicateur", value_name="montant")

fig = px.line(
    melted1, x="month", y="montant", color="indicateur",
    title="CA et coûts mensuel",
    labels={"month": "Mois", "montant": "Montant (€)", "indicateur": "Indicateur"},
    markers=True,
)


fig.show()

fig2 = px.line(df ,x="month", y="balance", title = "Bénéfices mensuels", 
               labels={"month": "Mois", "balance": "Montant (€)"})

fig2.show()

global_benef = load("silver_global_benef")["value"].iloc[0]
fig3 = go.Figure(go.Indicator(
    mode="number",
    value=global_benef,
    number={"prefix": "€", "valueformat": ",.0f"},
    title={"text": "Bénéfice annuel global"},
))
fig3.show()


## 9. Bénéfice par produit dans le temps
Table `silver_benef_produit_month`

In [ ]:
df = load("silver_benef_produit_month")
df["month"] = pd.to_datetime(df["month"])
produit = load("bronze_produit")



In [78]:
df

,month,product_id,balance,_silver_load_timestamp_utc
0,2025-01-01,1,27972.37,2026-08-11T07:54:51.311401+00:00
1,2025-01-01,2,43208.99,2026-08-11T07:54:51.311401+00:00
2,2025-01-01,3,7616.40,2026-08-11T07:54:51.311401+00:00
3,2025-01-01,4,47342.33,2026-08-11T07:54:51.311401+00:00
4,2025-01-01,5,39561.13,2026-08-11T07:54:51.311401+00:00
5,2025-01-01,6,116947.03,2026-08-11T07:54:51.311401+00:00
6,2025-02-01,1,185975.16,2026-08-11T07:54:51.311401+00:00
7,2025-02-01,4,35178.23,2026-08-11T07:54:51.311401+00:00
8,2025-02-01,5,153461.14,2026-08-11T07:54:51.311401+00:00
9,2025-02-01,6,16526.59,2026-08-11T07:54:51.311401+00:00


In [ ]:
produit[['product_id', 'product_name']]

,product_id,product_name,category,unit_cost_eur
0,1,Servo moteur industriel,Mechanical,420
1,2,Carte de contrôle,Electronic,180
2,3,Pompe hydraulique,Hydraulic,360
3,4,Module capteur,Electronic,95
4,5,Vanne haute pression,Hydraulic,240
...,...,...,...,...
67,2,Carte de contrôle,Electronic,180
68,3,Pompe hydraulique,Hydraulic,360
69,4,Module capteur,Electronic,95
70,5,Vanne haute pression,Hydraulic,240


In [ ]:
df = load("silver_benef_produit_month")
df["month"] = pd.to_datetime(df["month"])
produit = load("bronze_produit")
fig = px.line(
    df, x="month", y="balance", color="product_id",
    title="Évolution mensuelle du bénéfice par produit (prix de vente - coûts)",
    labels={"month": "Mois", "balance": "Bénéfice (€)", "product_id": "Produit"},
    markers=True,
)
fig.show()


## 10. Machines sous surveillance (alertes capteurs / caméras)
Table `silver_machines`

In [54]:
df = load("silver_machines")

fig = px.scatter(
    df, x="number_of_sensors_default", y="camera_event_id",
    text="machine_id",
    title="Alertes capteurs vs. événements caméra par machine",
    labels={"number_of_sensors_default": "Alertes capteurs (nb)", "camera_event_id": "Événements caméra (nb)"},
    size="number_of_sensors_default",
)
fig.update_traces(textposition="top center")
fig.show()


## 11. Incidents critiques (sévérité HIGH)
Tables `silver_high_severity_total` (fix requis) et `silver_to_resolve_count`

In [58]:
df_total

,severity,resolution_time_min,_silver_load_timestamp_utc
0,15,4241,2026-08-11T07:25:33.942838+00:00
1,18,6907,2026-08-11T07:25:33.942838+00:00
2,20,6674,2026-08-11T07:25:33.942838+00:00
3,17,5718,2026-08-11T07:25:33.942838+00:00
4,18,7444,2026-08-11T07:25:33.942838+00:00
5,14,5522,2026-08-11T07:25:33.942838+00:00
6,13,4701,2026-08-11T07:25:33.942838+00:00
7,17,4778,2026-08-11T07:25:33.942838+00:00
8,17,7061,2026-08-11T07:25:33.942838+00:00
9,16,5395,2026-08-11T07:25:33.942838+00:00


In [59]:
df_total = load("silver_high_severity_total")

fig = px.bar(
    df_total.sort_values("severity", ascending=False).head(15),
    x="machine_id", y="severity",
    title="Nb d'incidents HIGH par machine (total)",
    labels={"machine_id": "Machine", "severity": "Nb incidents HIGH"},
)
fig.show()

df_open = load("silver_to_resolve_count")

fig2 = px.bar(
    df_open.sort_values("resolution_time_min", ascending=False).head(15),
    x="machine_id", y="resolution_time_min",
    title="Temps de résolution cumulé — incidents HIGH non résolus",
    labels={"machine_id": "Machine", "resolution_time_min": "Temps de résolution (min)"},
    color="severity" if "severity" in df_open.columns else None,
)
fig2.show()


## 12. Évolution du nombre de pannes par sévérité
Tables `silver_nb_pannes_global/high/medium/low` (fix requis)

In [61]:
load("silver_nb_pannes_high")

,machine_id,_silver_load_timestamp_utc
0,19,2026-08-11T07:47:13.214908+00:00
1,31,2026-08-11T07:47:13.214908+00:00
2,26,2026-08-11T07:47:13.214908+00:00
3,33,2026-08-11T07:47:13.214908+00:00
4,34,2026-08-11T07:47:13.214908+00:00
5,20,2026-08-11T07:47:13.214908+00:00
6,38,2026-08-11T07:47:13.214908+00:00
7,34,2026-08-11T07:47:13.214908+00:00
8,33,2026-08-11T07:47:13.214908+00:00
9,43,2026-08-11T07:47:13.214908+00:00


In [64]:
def load_pannes(table, label):
    d = load(table)
    d["event_timestamp"] = pd.to_datetime(d["event_timestamp"])
    d = d.rename(columns={"machine_id": "nb_pannes"})
    d["severite"] = label
    return d[["event_timestamp", "nb_pannes", "severite"]]

pannes = pd.concat([
    load_pannes("silver_nb_pannes_high", "HIGH"),
    load_pannes("silver_nb_pannes_medium", "MEDIUM"),
    load_pannes("silver_nb_pannes_low", "LOW"),
])

fig = px.line(
    pannes, x="event_timestamp", y="nb_pannes", color="severite",
    title="Nombre de pannes par mois et par sévérité",
    labels={"event_timestamp": "Mois", "nb_pannes": "Nb de pannes", "severite": "Sévérité"},
    markers=True,
)
fig.show()


## 13. Alertes capteurs dans le temps
Table `silver_alerts_machines` (fix requis)

In [68]:
df

,timestamp,machine_id,temperature_c,vibration_level,pressure_bar,_silver_load_timestamp_utc
0,2024-07-01 13:40:43,M-1,-20.00,5.96,9.21,2026-08-11T07:54:51.311401+00:00
1,2024-04-11 04:22:21,M-1,66.69,2.93,13.86,2026-08-11T07:54:51.311401+00:00
2,2024-04-26 21:15:24,M-1,83.14,4.26,14.35,2026-08-11T07:54:51.311401+00:00
3,2024-05-26 13:55:26,M-1,74.09,4.07,13.14,2026-08-11T07:54:51.311401+00:00
4,2024-10-13 13:59:33,M-1,150.00,4.15,8.02,2026-08-11T07:54:51.311401+00:00
...,...,...,...,...,...,...
161,2024-12-18 19:01:12,M-9,52.98,4.20,14.90,2026-08-11T07:54:51.311401+00:00
162,2024-12-20 13:47:01,M-9,68.60,5.70,13.91,2026-08-11T07:54:51.311401+00:00
163,2024-01-29 03:22:35,M-9,82.78,4.41,15.52,2026-08-11T07:54:51.311401+00:00
164,2024-09-15 10:08:52,M-9,150.00,5.79,14.80,2026-08-11T07:54:51.311401+00:00


In [69]:
df = load("silver_alerts_machines")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["vibration_level"] = df["vibration_level"].fillna(0)
fig = px.scatter(
    df, x="timestamp", y="temperature_c", color="machine_id",
    size="vibration_level",
    title="Alertes température (taille = niveau de vibration)",
    labels={"timestamp": "Date", "temperature_c": "Température (°C)", "machine_id": "Machine"},

)
fig.show()

fig2 = px.scatter(
    df, x="timestamp", y="pressure_bar", color="machine_id",
    title="Alertes pression dans le temps",
    labels={"timestamp": "Date", "pressure_bar": "Pression (bar)", "machine_id": "Machine"},
)
fig2.show()


## 14. Coût caché des pannes (manque à gagner)
`manque_panne` — table `silver_monthly_mttr`

In [71]:
df

,month,factory_id,mttr_hours,revenue_global_eur,avg_revenue_hour,manque_panne,_silver_load_timestamp_utc
0,2025-01-01,1,4.78,737758.25,1010.627740,4830.800596,2026-08-11T07:54:51.311401+00:00
1,2025-01-01,2,3.09,737758.25,1010.627740,3122.839716,2026-08-11T07:54:51.311401+00:00
2,2025-01-01,3,5.05,737758.25,1010.627740,5103.670086,2026-08-11T07:54:51.311401+00:00
3,2025-01-01,4,5.99,737758.25,1010.627740,6053.660161,2026-08-11T07:54:51.311401+00:00
4,2025-01-01,5,1.80,737758.25,1010.627740,1819.129932,2026-08-11T07:54:51.311401+00:00
5,2025-02-01,1,4.63,1166386.12,1597.789205,7397.764021,2026-08-11T07:54:51.311401+00:00
6,2025-02-01,2,3.24,1166386.12,1597.789205,5176.837026,2026-08-11T07:54:51.311401+00:00
7,2025-02-01,3,1.35,1166386.12,1597.789205,2157.015427,2026-08-11T07:54:51.311401+00:00
8,2025-02-01,4,2.41,1166386.12,1597.789205,3850.671985,2026-08-11T07:54:51.311401+00:00
9,2025-02-01,5,1.93,1166386.12,1597.789205,3083.733167,2026-08-11T07:54:51.311401+00:00


In [73]:
df2

,factory_id,factory_name,country,city,machine_count
0,1,Usine Lyon,France,Lyon,20
1,2,Usine Lille,France,Lille,18
2,3,Usine Toulouse,France,Toulouse,22
3,4,Usine Hamburg,Germany,Hamburg,25
4,5,Usine Turin,Italy,Turin,19
5,1,Usine Lyon,France,Lyon,20
6,2,Usine Lille,France,Lille,18
7,3,Usine Toulouse,France,Toulouse,22
8,4,Usine Hamburg,Germany,Hamburg,25
9,5,Usine Turin,Italy,Turin,19


In [74]:
df = load("silver_monthly_mttr")
df2= load("bronze_usine")

df["month"] = pd.to_datetime(df["month"])
df = df.merge(df2[["factory_id", "factory_name"]], on="factory_id", how="left")
fig = px.bar(
    df, x="month", y="manque_panne", color="factory_name",
    title="Manque à gagner estimé lié aux pannes (MTTR × CA horaire moyen)",
    labels={"month": "Mois", "manque_panne": "Manque à gagner (€)", "factory_name": "Usine"},
)
fig.show()

fig2 = px.line(
    df.groupby("month", as_index=False)["mttr_hours"].sum(),
    x="month", y="mttr_hours",
    title="Total heures d'arrêt (MTTR) par mois — toutes usines",
    labels={"month": "Mois", "mttr_hours": "Heures d'arrêt"},
    markers=True,
)
fig2.show()
